# **RAG del Gemma2 finetunado**

En este cuaderno hemos montado un sistema de RAG relativamente sencillo, siguiendo las indicaciones de Gemini que, por cierto, da resultados relativamente aceptables teniendo en cuenta lo sencillo que es. Lo hemos hecho algo más sofisticado, eso sí comparado con el que presentamos en la práctica de LLM Engineering.

Lo que es mejor es que se basa en un código perfectamente entendible, a pesar de que hace uso de embaddings y faiss, es decir, todo lo contrario a lo que nos dio DeepSeek.

Empezamos instalamos Unsloth

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

Instalamos faiss-cpu, ya que no fue posible hacer lo propio con la versión de gpu.

In [ ]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.5 MB/s eta 0:00:00


Cargamos el modelo ya fintuneado de 4 bits con Unsloth.

Lo cierto es que ahora este paso no es tan importante, porque haremos un análisis del RAG mediante RAGAS en un cuaderno posterior.

In [ ]:
from unsloth import FastLanguageModel
import torch

# La ruta donde están tus 4 ficheros de modelo y el tokenizador
model_path = "/content/drive/MyDrive/Practica_Bootcamp_IA_4/modelo-Gemma_2-9b_2011-19_finetuned-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # Unsloth lee la carpeta y une los shards
    max_seq_length = 2048,   # El mismo que usaste en el entrenamiento
    dtype = None,            # Auto-detección (usará float16/bfloat16)
    load_in_4bit = True,     # <--- AQUÍ activamos la magia del 4bit para inferencia
)

# Ponemos el modelo en modo inferencia para que no consuma memoria de gradientes
FastLanguageModel.for_inference(model)

print("¡Modelo Gemma-2 electoral cargado exitosamente en 4 bits!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: If you want to finetune Gemma 2, install flash-attn to make it faster!
To install flash-attn, do the below:

pip install --no-deps --upgrade "flash-attn>=2.6.3"
==((====))==  Unsloth 2026.2.1: Fast Gemma2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

¡Modelo Gemma-2 electoral cargado exitosamente en 4 bits!


Ahora cargamos el datset original de los datos electorales, de las 5 elecciones, unificado y curado.

In [ ]:
import pandas as pd

In [ ]:
strings = {'Sección' : 'str', 'cod_ccaa' : 'str', 'cod_prov' : 'str', 'cod_mun' : 'str', 'cod_sec' : 'str', 'Distrito' : 'str'}

df = pd.read_csv('/content/drive/MyDrive/Practica_Bootcamp_IA_4/eleccion_unificado.csv', dtype = strings)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 181092 entries, 0 to 181091
Data columns (total 76 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Elecciones                        181092 non-null  object 
 1   Sección                           181092 non-null  object 
 2   cod_ccaa                          181092 non-null  object 
 3   cod_prov                          181092 non-null  object 
 4   cod_mun                           181092 non-null  object 
 5   cod_sec                           181092 non-null  object 
 6   CCAA                              181092 non-null  object 
 7   Provincia                         181092 non-null  object 
 8   Municipio                         181092 non-null  object 
 9   Distrito                          181092 non-null  object 
 10  Censo_Esc                         181092 non-null  int64  
 11  Votos_Total                       181092 non-null  i

In [ ]:
df['Elecciones'].unique()

array(['Abril 2019', 'Noviembre 2019', 'Noviembre 2011', 'Diciembre 2015',
       'Junio 2016'], dtype=object)

In [ ]:
len(df['Municipio'].unique())

8118

Ahora vamos con el RAG en sí.

Esta celda hace varias cosas:

- Carga el embedder de la librería transformers
- Prepara una columna en el dataset que reune varios datos de cada fila
- Hace el embedding de cada elemento de esta columna
- Crea el índice FAISS para agilizar las búsquedas

Lo cierto es que en el uso del modelo hubo que potenciar el peso de la fecha de las elecciones, pues el embedding daba preferencia a los municipios, un fenómeno que, según Gemini, dilución semántica.

Posteriormente añadimos referencias a los municipios y provincias para intentar, más o menos, que los embeddings no se parecieran demasiado entre sí. Tuvimos, como veremos en el cuadendo del RAGAS, un éxito solo relativo.

In [ ]:
def crear_narrativa(row):

    return (f"elecciones: {row['Elecciones']}. "
            f"municipio: {row['Municipio']}. "
            f"provincia: {row['Provincia']}. "
            f"elecciones: {row['Elecciones']}. "
            f"municipio: {row['Municipio']}. "
            f"provincia: {row['Provincia']}. "
            f"elecciones: {row['Elecciones']}. "
            f"municipio: {row['Municipio']}. "
            f"elecciones: {row['Elecciones']}. "


            f"Resultados electorales de {row['Elecciones']} en la sección {row['cod_sec']} del distrito {row['Distrito']} en el municipio de {row['Municipio']} de la provincia {row['Provincia']}. "
            f"La participación electoral fue del {row['Participación']:.1%}. "
            f"El partido más votado fue el {row['Ganador']}, el segundo fue el {row['Segundo']}, el tercero fue el {row['Tercero']}, el cuarto fue el {row['Cuarto']}, y el quinto fue el {row['Quinto']} ."
            f"Contexto socioeconómico: Renta media por persona de {row['Renta persona 2017']} euros, con un porcentaje de afiliados a la Seguridad Social respecto a la población del {row['% Afiliados SS / Población']:.2%}. "
            f"Contexto demográfico: La sección tiene un porcentaje de mayores de 65 años del {row['% mayores 65 años']:.1%}, y un porcentaje de menores de 19 años del {row['% menores 19 años']:.1%}. "
            f"Perfil de voto: PP({row['% PP']:.2%}), PSOE({row['% PSOE']:.2%}), VOX({row['% VOX']:.2%}), Cs({row['% Cs']:.2%}), IU({row['% IU']:.2%}), UP({row['% UP']:.2%})." )

df['metadata_vectorial'] = df.apply(crear_narrativa, axis=1)

Creamos otra columna en el dataset más sencilla, destinada, si acaso al contexto del modelo más que a la creación de los embeddings.

In [ ]:
def crear_contexto(row):


  return (f"Resultados electorales de {row['Elecciones']} en la sección {row['cod_sec']} del distrito {row['Distrito']} en el municipio de {row['Municipio']} de la provincia {row['Provincia']}. "
          f"La participación electoral fue del {row['Participación']:.1%}. "
          f"El partido más votado fue el {row['Ganador']}, el segundo fue el {row['Segundo']}, el tercero fue el {row['Tercero']}, el cuarto fue el {row['Cuarto']}, y el quinto fue el {row['Quinto']} . "
          f"Contexto socioeconómico: Renta media por persona de {row['Renta persona 2017']} euros, con un porcentaje de afiliados a la Seguridad Social respecto a la población del {row['% Afiliados SS / Población']:.2%}. "
          f"Contexto demográfico: La sección tiene un porcentaje de mayores de 65 años del {row['% mayores 65 años']:.1%}, y un porcentaje de menores de 19 años del {row['% menores 19 años']:.1%}. "
          f"Perfil de voto: PP({row['% PP']:.2%}), PSOE({row['% PSOE']:.2%}), VOX({row['% VOX']:.2%}), Cs({row['% Cs']:.2%}), IU({row['% IU']:.2%}), UP({row['% UP']:.2%}).")

In [ ]:
df['contexto'] = df.apply(crear_contexto, axis=1)

In [ ]:
df['metadata_vectorial'][0]

'elecciones: Abril 2019. municipio: Abla. provincia: Almería. elecciones: Abril 2019. municipio: Abla. provincia: Almería. elecciones: Abril 2019. municipio: Abla. elecciones: Abril 2019. Resultados electorales de Abril 2019 en la sección 0400101001 del distrito 01 en el municipio de Abla de la provincia Almería. La participación electoral fue del 75.7%. El partido más votado fue el PSOE, el segundo fue el PP, el tercero fue el Cs, el cuarto fue el VOX, y el quinto fue el UP .Contexto socioeconómico: Renta media por persona de 9159.0 euros, con un porcentaje de afiliados a la Seguridad Social respecto a la población del 23.30%. Contexto demográfico: La sección tiene un porcentaje de mayores de 65 años del 27.0%, y un porcentaje de menores de 19 años del 14.0%. Perfil de voto: PP(19.53%), PSOE(42.73%), VOX(11.53%), Cs(17.17%), IU(0.00%), UP(5.77%).'

In [ ]:
df['contexto'][0]

'Resultados electorales de Abril 2019 en la sección 0400101001 del distrito 01 en el municipio de Abla de la provincia Almería. La participación electoral fue del 75.7%. El partido más votado fue el PSOE, el segundo fue el PP, el tercero fue el Cs, el cuarto fue el VOX, y el quinto fue el UP . Contexto socioeconómico: Renta media por persona de 9159.0 euros, con un porcentaje de afiliados a la Seguridad Social respecto a la población del 23.30%. Contexto demográfico: La sección tiene un porcentaje de mayores de 65 años del 27.0%, y un porcentaje de menores de 19 años del 14.0%. Perfil de voto: PP(19.53%), PSOE(42.73%), VOX(11.53%), Cs(17.17%), IU(0.00%), UP(5.77%).'

Ahora, por fin, creamos los embeddings y el índice RAG, tras descargar el modelo correspondente de SentenceTransformer y la librería Faiss.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Cargamos el modelo que convierte texto en vectores (ligero y potente)
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# 2. Preparamos los textos para indexar
# Creamos una columna que resuma la fila para que el buscador la entienda
# df['resumen_busqueda'] = df.apply(lambda x: f"Municipio: {x['Municipio']}, Elecciones: {x['Elecciones']}, Provincia: {x['Provincia']}", axis=1)

# df['resumen_busqueda'] = df.apply(lambda x:
  #  f"ELECCIONES {x['Elecciones']}. FECHA: {x['Elecciones']}. LUGAR: {x['Municipio']}, {x['Provincia']}. DATOS DE {x['Elecciones']}.",
  #  axis=1)

# 3. Convertimos a vectores (esto tardará unos minutos por las 181k filas)
embeddings = embedder.encode(df['metadata_vectorial'].tolist(), show_progress_bar=True)


# 4. Creamos el índice FAISS para búsqueda ultra rápida
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("¡Índice RAG creado con éxito!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5660 [00:00<?, ?it/s]

¡Índice RAG creado con éxito!


Ahora guardamos el fichero faiss y los metadata, que usaremos posteriormente en el RAGAS.

In [ ]:
import faiss
import numpy as np
import pickle

# 1. Guardar el índice de FAISS
faiss.write_index(index, "/content/drive/MyDrive/Practica_Bootcamp_IA_4/electoral_index_2011-19.faiss")

# 2. Guardar los embeddings crudos (por si acaso) y los textos asociados
# Es vital guardar el dataframe o la columna de textos para que los índices coincidan
df[['metadata_vectorial']].to_pickle("/content/drive/MyDrive/Practica_Bootcamp_IA_4/metadata_rows_2011-19.pkl")

print("¡Índice y metadata guardados!")

¡Índice y metadata guardados!


Comprobamos que tenemos las columnas de metadata y contexto en el dataset, y lo guardamos.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 181092 entries, 0 to 181091
Data columns (total 78 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Elecciones                        181092 non-null  object 
 1   Sección                           181092 non-null  object 
 2   cod_ccaa                          181092 non-null  object 
 3   cod_prov                          181092 non-null  object 
 4   cod_mun                           181092 non-null  object 
 5   cod_sec                           181092 non-null  object 
 6   CCAA                              181092 non-null  object 
 7   Provincia                         181092 non-null  object 
 8   Municipio                         181092 non-null  object 
 9   Distrito                          181092 non-null  object 
 10  Censo_Esc                         181092 non-null  int64  
 11  Votos_Total                       181092 non-null  i

In [ ]:
df.to_csv('/content/drive/MyDrive/Practica_Bootcamp_IA_4/eleccion_unificado_2011-19_metadata.csv', index=False)

Esta función recoge la pregunta, que pasa por el embedder y FAISS busca las filas más parecidas. Se añade un contexto, y se manda al modelo, que contesta, y la función finalmente devuelve la respuesta parseada.

Se trata de un ejemplo que reconocemos que tiene un interés limitado, ya que en el siguiente cuaderno examinaremos el RAG mediante RAGAS, pero ahí va.

In [ ]:
from sentence_transformers import CrossEncoder

# Este modelo puntúa la relación real entre la Pregunta y el Documento
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def agente_recuperador(pregunta, top_k_inicial=30, top_k_final=6):
    # 1. Búsqueda inicial en FAISS
    query_vector = embedder.encode([pregunta]).astype('float32')
    _, indices = index.search(query_vector, top_k_inicial)
    candidatos = df.iloc[indices[0]]

    # 2. Re-Ranking con Cross-Encoder
    # Preparamos los pares (Pregunta, Texto del Candidato)
    pares = [[pregunta, texto] for texto in candidatos['metadata_vectorial'].tolist()]
    scores = cross_model.predict(pares)

    # 3. Ordenamos por puntuación y seleccionamos los mejores
    candidatos['score'] = scores
    finalistas = candidatos.sort_values(by='score', ascending=False).head(top_k_final)

    return finalistas

In [ ]:
finalistas = agente_recuperador("¿Cuál fue la participación electoral en el municipio de Bargas provincia de Toledo en las elecciones de Abril 2019?")
print(finalistas)

           Elecciones                Sección cod_ccaa cod_prov cod_mun  \
12238      Abril 2019  022019041074501901004       07       45   45019   
12236      Abril 2019  022019041074501901002       07       45   45019   
12235      Abril 2019  022019041074501901001       07       45   45019   
12239      Abril 2019  022019041074501901005       07       45   45019   
12237      Abril 2019  022019041074501901003       07       45   45019   
48557  Noviembre 2019  022019111074501901003       07       45   45019   

          cod_sec                  CCAA Provincia Municipio Distrito  ...  \
12238  4501901004  Castilla - La Mancha    Toledo    Bargas       01  ...   
12236  4501901002  Castilla - La Mancha    Toledo    Bargas       01  ...   
12235  4501901001  Castilla - La Mancha    Toledo    Bargas       01  ...   
12239  4501901005  Castilla - La Mancha    Toledo    Bargas       01  ...   
12237  4501901003  Castilla - La Mancha    Toledo    Bargas       01  ...   
48557  4501901003  

/tmp/ipython-input-3984310648.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidatos['score'] = scores


In [ ]:
def sistema_rag_electoral(pregunta, top_k_inicial=30, top_k_final=10):
    # 1. Vectorizar la pregunta del usuario
    #query_vector = embedder.encode([pregunta]).astype('float32')

    # 2. Buscar en el espacio vectorial
    #distancias, indices = index.search(query_vector, top_k)

    # 3. Recuperar las filas correspondientes del DataFrame original
    #filas_recuperadas = df.iloc[indices[0]]

    filas_recuperadas = agente_recuperador(pregunta, top_k_inicial=30, top_k_final=10)

    # 4. Construir el contexto
    contexto = "Datos recuperados del histórico electoral:\n"
    for _, fila in filas_recuperadas.iterrows():
        contexto += f"{fila['contexto']} \n"

        #contexto += f"- En {fila['Municipio']} ({fila['Elecciones']}): "
        #contexto += f"porcentage PP: {fila['% PP']:.2%}, porcentage PSOE: {fila['% PSOE']:.2%}, Renta persona 2017: {fila['Renta persona 2017']}€.\n"

    # 5. Crear el prompt final para Gemma-2
    prompt_final = f"""### Instrucción: Eres un analista de datos experto. Utilizando exclusivamente el contexto proporcionado, responde a la pregunta del usuario.
IMPORTANTE: No expliques cómo se hace el análisis, REALIZA el análisis cuantitativo directamente usando los números del contexto. Si hay varias secciones, compáralas y extrae una conclusión basada en las cifras de participación y renta.
Cuando compares datos de distintas secciones, utiliza una tabla de Markdown para mayor claridad.

{pregunta}

### Contexto:
{contexto}

### Respuesta:"""

    print(prompt_final)

    # 6. Inferencia con tu modelo de 4 bits
    inputs = tokenizer([prompt_final], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 600, use_cache = True)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("### Respuesta:")[-1].strip()

In [ ]:
# resultado = sistema_rag_electoral("Municipio de Santander, provincia de Cantabria, elecciones Noviembre 2019")

resultado = sistema_rag_electoral("¿Elecciones: Abril 2019: Cómo se relaciona la participación electoral con la renta personal en el municipio de Bargas en la provincia de Toledo en las elecciones de Abril 2019?")
print(resultado)

/tmp/ipython-input-3984310648.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidatos['score'] = scores
Unsloth: Input IDs of shape torch.Size([1, 2405]) with length 2405 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


### Instrucción: Eres un analista de datos experto. Utilizando exclusivamente el contexto proporcionado, responde a la pregunta del usuario. 
IMPORTANTE: No expliques cómo se hace el análisis, REALIZA el análisis cuantitativo directamente usando los números del contexto. Si hay varias secciones, compáralas y extrae una conclusión basada en las cifras de participación y renta.
Cuando compares datos de distintas secciones, utiliza una tabla de Markdown para mayor claridad.

¿Elecciones: Abril 2019: Cómo se relaciona la participación electoral con la renta personal en el municipio de Bargas en la provincia de Toledo en las elecciones de Abril 2019?

### Contexto:
Datos recuperados del histórico electoral:
Resultados electorales de Abril 2019 en la sección 4518201001 del distrito 01 en el municipio de Las Ventas con Peña Aguilera de la provincia Toledo. La participación electoral fue del 82.7%. El partido más votado fue el PSOE, el segundo fue el PP, el tercero fue el VOX, el cuarto fue el

In [ ]:
print(resultado.replace('**', ''))

| Sección | Participación | Renta Media (€) |
|---|---|---|
| 4518201001 | 82.7% | 8446.0 |
| 4518202001 | 82.0% | 8556.0 |
| 4501801001 | 72.4% | 7075.0 |
| 4501901003 | 73.9% | 9530.0 |
| 4501901005 | 86.1% | 16403.0 |
| 4501901002 | 70.9% | 9378.0 |
| 4501901001 | 76.9% | 9969.0 |
| 4501901004 | 75.6% | 9269.0 |
| 4516301001 | 81.0% | 8100.0 |


Se observa una correlación positiva entre la renta media por persona y la participación electoral en las secciones analizadas. Las secciones con mayor renta media (4501901005, 4501901001, 4501901004) presentan también las tasas de participación más altas.
